# 🤖 Vayu — Face Data Upload Pipeline

This notebook processes person images, generates **Facenet512** embeddings, stores them in a **FAISS** index, and records metadata in **SQLite**.

---

### 📁 Expected Folder Structure on Google Drive

```
My Drive/J.A.R.V.I.S./persons1/
  ├── aryan/
  │     ├── image1.jpg
  │     ├── image2.png
  │     └── description.txt
  ├── sarika/
  │     ├── photo.jpeg
  │     └── description.txt
  └── ...
```

### 🚀 How to Run
1. **Run Cell 1** — Installs dependencies & restarts the runtime automatically.
2. **Run all remaining cells** after the runtime restarts.
3. Download `face_index.faiss` and `face_database.db` from your Drive when done.

> ⚡ **Tip:** Use GPU runtime for faster processing (Runtime ➜ Change runtime type ➜ GPU)

---
## 📦 Cell 1 — Install Dependencies & Restart Runtime
Run this cell **first**. It will install packages and auto-restart the runtime to avoid numpy binary conflicts.

In [ ]:
import subprocess, sys, os

if not os.path.exists("/tmp/_vayu_deps_installed"):
    packages = [
        "numpy<2.0",
        "deepface",
        "faiss-cpu",
        "tf-keras",
    ]
    for pkg in packages:
        print(f"📦 Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    open("/tmp/_vayu_deps_installed", "w").close()
    print("\n✅ All packages installed! Restarting runtime...")
    os.kill(os.getpid(), 9)
else:
    print("✅ Dependencies already installed, ready to go!")

---
## 🔗 Cell 2 — Mount Google Drive & Configure Paths
After runtime restarts, run this cell to mount your Drive and set the input/output paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════
#  ⚙️  CONFIGURATION — Change these paths to match YOUR Drive
# ═══════════════════════════════════════════════════════

PERSONS_FOLDER = "/content/drive/MyDrive/J.A.R.V.I.S./persons1"
OUTPUT_DIR     = "/content/drive/MyDrive/J.A.R.V.I.S./persons1"

FAISS_INDEX_PATH = f"{OUTPUT_DIR}/face_index.faiss"
SQLITE_DB_PATH   = f"{OUTPUT_DIR}/face_database.db"

print(f"📂 Persons folder : {PERSONS_FOLDER}")
print(f"💾 FAISS index    : {FAISS_INDEX_PATH}")
print(f"💾 SQLite DB      : {SQLITE_DB_PATH}")

---
## 🧠 Cell 3 — DeepFace Embedding Generator
Generates normalized 512-d Facenet512 embeddings using MTCNN face detection.

In [ ]:
from deepface import DeepFace
import numpy as np

def front_face_embedding_value(img_path: str):
    """
    Generate a normalized 512-d Facenet512 embedding for the front (largest) face
    detected in the image using MTCNN.

    Returns:
        np.ndarray of shape (512,) on success, or empty list [] on failure.
    """
    try:
        print(f"  🔍 Generating embedding for {img_path}...")
        embedding_obj = DeepFace.represent(
            img_path=img_path,
            model_name="Facenet512",
            enforce_detection=True,
            detector_backend="mtcnn",
        )

        if len(embedding_obj) > 1:
            front_face = max(
                embedding_obj,
                key=lambda x: x["facial_area"]["w"] * x["facial_area"]["h"],
            )
            front_face_embedding = np.array(front_face["embedding"], dtype=np.float32)
            print(f"  ℹ️ Found {len(embedding_obj)} faces, using the largest one.")
        else:
            front_face_embedding = np.array(
                embedding_obj[0]["embedding"], dtype=np.float32
            )

        # L2-normalize so inner product == cosine similarity
        front_face_embedding = front_face_embedding / np.linalg.norm(front_face_embedding)
        return front_face_embedding

    except Exception as e:
        print(f"  ❌ Error generating embedding: {e}")
        return []

---
## 📊 Cell 4 — FAISS Vector Database
Manages the FAISS index for storing and retrieving face embeddings using cosine similarity.

In [ ]:
import faiss
import os

class FAISS_VectorDB:
    def __init__(self, faiss_index_path: str):
        self.faiss_index_path = faiss_index_path
        self.embedding_dim = 512  # Facenet512 produces 512-d embeddings
        self.index = self._init_faiss_index()

    def _init_faiss_index(self):
        """Load existing FAISS index or create a new one."""
        if os.path.exists(self.faiss_index_path):
            index = faiss.read_index(self.faiss_index_path)
            print(f"✅ Loaded existing FAISS index from {self.faiss_index_path}")
        else:
            base_index = faiss.IndexFlatIP(self.embedding_dim)
            index = faiss.IndexIDMap(base_index)
            print("✅ Created new FAISS index (cosine similarity via Inner Product)")
        return index

    def insert_data(self, image_paths: list, next_id: int) -> list:
        """Generate embeddings for images and add them to the FAISS index."""
        ids = []
        embeddings = []

        for image_path in image_paths:
            embedding = front_face_embedding_value(image_path)

            if embedding is None or len(embedding) == 0:
                print(f"  ⚠️ Skipping {image_path} (no embedding)")
                continue

            ids.append(next_id)
            embeddings.append(embedding)
            next_id += 1

        if embeddings:
            embeddings_array = np.array(embeddings, dtype=np.float32)
            ids_array = np.array(ids, dtype=np.int64)
            self.index.add_with_ids(embeddings_array, ids_array)
            print(f"  ✅ Added {len(ids)} embedding(s). Last FAISS ID = {next_id - 1}")

        return ids

    def save_faiss_index(self):
        """Persist the FAISS index to disk."""
        faiss.write_index(self.index, self.faiss_index_path)
        print(f"💾 FAISS index saved → {self.faiss_index_path}")

---
## 🗃️ Cell 5 — SQLite Metadata Database
Stores person names, descriptions, and links them to their FAISS embedding IDs.

In [ ]:
import sqlite3

class SQLite_SqlDB:
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        """Create the persons + face_embeddings_id tables if they don't exist."""
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("PRAGMA foreign_keys = ON;")

        cur.execute("""
            CREATE TABLE IF NOT EXISTS persons (
                person_id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                description TEXT
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS face_embeddings_id (
                faiss_id INTEGER PRIMARY KEY,
                person_id INTEGER NOT NULL,
                FOREIGN KEY (person_id)
                    REFERENCES persons (person_id)
                    ON DELETE CASCADE
                    ON UPDATE CASCADE
            );
        """)

        conn.commit()
        cur.close()
        conn.close()
        print(f"✅ SQLite database initialized → {self.db_path}")

    def max_faiss_id(self) -> int:
        """Return the maximum FAISS ID in the DB (0 if empty)."""
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("SELECT MAX(faiss_id) FROM face_embeddings_id")
        result = cur.fetchone()[0]
        cur.close()
        conn.close()
        return result if result else 0

    def insert_person(self, faiss_ids: list, name: str, description: str):
        """Insert a person and link their FAISS IDs."""
        conn = sqlite3.connect(self.db_path)
        cur = conn.cursor()
        cur.execute("PRAGMA foreign_keys = ON;")

        cur.execute(
            "INSERT INTO persons(name, description) VALUES (?, ?)",
            (name, description),
        )
        person_id = cur.lastrowid

        data = [(fid, person_id) for fid in faiss_ids]
        cur.executemany(
            "INSERT INTO face_embeddings_id (faiss_id, person_id) VALUES (?, ?)", data
        )

        conn.commit()
        cur.close()
        conn.close()
        print(f"  ✅ Added '{name}' to SQLite (person_id={person_id})")

---
## ▶️ Cell 6 — Run the Pipeline
Processes all person subfolders, generates embeddings, and saves everything to Drive.

In [ ]:
def run_pipeline(persons_folder: str, faiss_path: str, db_path: str):
    """
    Process all person sub-folders:
      1. Read images from each sub-folder
      2. Generate Facenet512 embeddings via DeepFace
      3. Store embeddings in FAISS index
      4. Store person name + description in SQLite
    """
    if not os.path.exists(persons_folder):
        print(f"❌ Folder '{persons_folder}' does not exist!")
        return

    # Initialize databases
    vector_db = FAISS_VectorDB(faiss_path)
    sql_db = SQLite_SqlDB(db_path)

    subfolders = sorted([
        f for f in os.listdir(persons_folder)
        if os.path.isdir(os.path.join(persons_folder, f))
    ])

    if not subfolders:
        print("⚠️ No sub-folders found in the persons folder.")
        return

    print(f"\n🚀 Found {len(subfolders)} person folder(s): {subfolders}\n")

    IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".webp"}

    for subfolder in subfolders:
        subfolder_path = os.path.join(persons_folder, subfolder)
        print(f"\n{'='*50}")
        print(f"👤 Processing: {subfolder}")
        print(f"{'='*50}")

        # Collect image paths
        image_paths = sorted([
            os.path.join(subfolder_path, f)
            for f in os.listdir(subfolder_path)
            if os.path.splitext(f.lower())[1] in IMAGE_EXTENSIONS
        ])
        print(f"  📸 Found {len(image_paths)} image(s)")

        if not image_paths:
            print(f"  ⚠️ No images found, skipping {subfolder}")
            continue

        # Read description
        description = ""
        desc_file = os.path.join(subfolder_path, "description.txt")
        if os.path.exists(desc_file):
            with open(desc_file, "r") as f:
                description = f.read().strip()
            print(f"  📝 Description: {description}")
        else:
            print(f"  ⚠️ No description.txt found")

        # Generate embeddings & store
        next_id = sql_db.max_faiss_id() + 1
        faiss_ids = vector_db.insert_data(image_paths, next_id)

        if faiss_ids:
            sql_db.insert_person(faiss_ids, subfolder, description)
        else:
            print(f"  ⚠️ No valid embeddings for {subfolder}, skipping DB insert.")

    # Save FAISS index
    vector_db.save_faiss_index()

    print(f"\n{'='*50}")
    print("🎉 Pipeline complete!")
    print(f"   FAISS index → {faiss_path}")
    print(f"   SQLite DB   → {db_path}")
    print(f"{'='*50}")
    print("\n📥 Download these two files and place them in your local project root.")


# ── Run it! ──
run_pipeline(PERSONS_FOLDER, FAISS_INDEX_PATH, SQLITE_DB_PATH)

---
## 🔍 Cell 7 — Verify Results (Optional)
Quick sanity check to see what got stored in the database.

In [ ]:
import sqlite3

conn = sqlite3.connect(SQLITE_DB_PATH)
cur = conn.cursor()

print("📋 Persons Table:")
print("-" * 60)
cur.execute("SELECT * FROM persons")
for row in cur.fetchall():
    print(f"  ID: {row[0]} | Name: {row[1]} | Desc: {row[2]}")

print(f"\n📋 Face Embeddings Table:")
print("-" * 60)
cur.execute("SELECT * FROM face_embeddings_id")
for row in cur.fetchall():
    print(f"  FAISS ID: {row[0]} | Person ID: {row[1]}")

cur.close()
conn.close()

print(f"\n✅ Total embeddings in FAISS index: {faiss.read_index(FAISS_INDEX_PATH).ntotal}")